# CliMaPan-Lab Quick Start

Run a minimal economic simulation and inspect the results using AMBER's DataFrame-backed agent system.

In [1]:
import warnings
warnings.filterwarnings('ignore')

from climapan_lab.base_params import economic_params
from climapan_lab.src.models import EconModel
import polars as pl
import numpy as np

## 1. Configure a small simulation

100 consumers, 3 consumer-goods firms, 2 capital-goods firms, 60 daily steps.

In [2]:
params = economic_params.copy()
params.update({
    'c_agents': 100,           # 100 consumers
    'capitalists': 10,         # 10 firm owners
    'csf_agents': 3,           # 3 consumer-goods firms
    'cpf_agents': 2,           # 2 capital-goods firms
    'steps': 60,               # 60 daily steps
    'seed': 42,
    'show_progress': False,
    'covid_settings': None,    # no pandemic
    'climateModuleFlag': False,
    'verboseFlag': False,
})

print(f'Agents: {params["c_agents"]} consumers + firms')
print(f'Steps: {params["steps"]} days')

Agents: 100 consumers + firms
Steps: 60 days


## 2. Run the model

`model.run()` returns a dict with `agents` (Polars DataFrame of all agents) and `model` (Polars DataFrame of monthly metrics).

In [3]:
model = EconModel(params)
result = model.run()

print(f'Completed in {result["info"]["run_time"]:.2f}s')
print(f'Steps: {result["info"]["steps"]}')
print(f'Agent rows: {result["agents"].height}')
print(f'Model columns: {len(result["model"].columns)}')

Completed in 0.14s
Steps: 60
Agent rows: 100
Model columns: 86


## 3. Inspect the agent population

AMBER stores all agents as a **single Polars DataFrame**. Each row is one agent, each column is an attribute. You can query it with Polars expressions directly.

In [4]:
agents_df = result['agents']
print(f'Shape: {agents_df.shape}')
print('First 5 agents:')
# Show key columns only
key_cols = ['id', 'consumptionSubsistenceLevel', 'worker_additional_consumption',
            'consumerType', 'deposit', 'wage', 'employed']
available = [c for c in key_cols if c in agents_df.columns]
print(agents_df.select(available).head(5))

Shape: (100, 139)
First 5 agents:
shape: (5, 139)
id | consumptionSubsistenceLevel | worker_additional_consumption | ... | consumerType | deposit
0  | 49.55                      | 5.09                          | ... | capitalists  | 7741.41
1  | 49.55                      | 5.09                          | ... | capitalists  | 6241.41
2  | 49.55                      | 5.09                          | ... | workers      | 2400.00
3  | 49.55                      | 5.09                          | ... | workers      | 2400.00
4  | 49.55                      | 5.09                          | ... | workers      | 2400.00


### Filter by consumer type using Polars

In [5]:
if 'consumerType' in agents_df.columns:
    workers = agents_df.filter(pl.col('consumerType') == 'workers')
    capitalists = agents_df.filter(pl.col('consumerType') == 'capitalists')
    print(f'Workers: {workers.height}')
    print(f'Capitalists: {capitalists.height}')
    print(f'Avg worker wage: {workers["wage"].mean():.2f}')
    print(f'Avg capitalist deposit: {capitalists["deposit"].mean():.2f}')

Workers: 53
Capitalists: 10
Avg worker wage: 1800.00
Avg capitalist deposit: 9855.54


## 4. Model-level metrics

The model records GDP, Gini coefficient, unemployment rate, and investment each month.

In [6]:
model_df = result['model']
key_cols = [c for c in ['GDP', 'UnemploymentRate', 'Gini', 'Investment'] if c in model_df.columns]
print('Final 5 months of model data:')
print(model_df.select(key_cols).drop_nulls().tail(5))

Final 5 months of model data:
shape: (5, 4)
GDP      | UnemploymentRate | Gini   | Investment
72297.29 | 0.44             | 0.45   | 7718.16
73847.54 | 0.43             | 0.46   | 7883.36


## 5. Using AMBER's view API on live agents

During a simulation you can access agent lists and query columns via `.getWage()`, `.getConsumerType()`, etc.

In [7]:
# Create a fresh model and inspect before running
p = economic_params.copy()
p.update({'steps': 5, 'c_agents': 20, 'seed': 42, 'show_progress': False,
          'covid_settings': None, 'climateModuleFlag': False})

m = EconModel(p)
m.setup()  # initialize but don't run the full simulation

print(f'Consumer agents: {len(m.consumer_agents)}')
print(f'CS firms: {len(m.csfirm_agents)}')
print(f'CP firms: {len(m.cpfirm_agents)}')

# Use AMBER view API to query agent columns
wages = m.consumer_agents.getWage()
print(f'Wages (first 5): {wages[:5].tolist()}')
print(f'Mean wage: {np.mean(wages):.2f}')

Consumer agents: 20
CS firms: 6
CP firms: 2
Wages (first 5): [1800.0, 1800.0, 1800.0, 1800.0, 1800.0]
Mean wage: 1800.00


## Summary

- **`model.run()`** returns Polars DataFrames — all standard Polars operations work
- **`model.consumer_agents.getWage()`** queries a column from all consumers
- **`model.consumer_agents[0]`** accesses an individual Python Agent object
- **Setting `agent.deposit = 9999`** on a Python Agent auto-syncs to the DataFrame
- Use **`pl.col('...')`** expressions to filter, aggregate, and transform agent data